In [1]:
!apt-get update -qq > /dev/null
!apt-get install -y -qq openjdk-11-jdk-headless > /dev/null
!wget -q https://archive.apache.org/dist/cassandra/4.1.5/apache-cassandra-4.1.5-bin.tar.gz
!tar -xzf apache-cassandra-4.1.5-bin.tar.gz
!pip install -q pyspark==3.5.3 cassandra-driver streamlit plotly

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 15.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


In [2]:
!ls /usr/lib/jvm/
!ls apache-cassandra-4.1.5/bin/cassandra
!pip show pyspark | grep Version

java-1.11.0-openjdk-amd64  java-1.21.0-openjdk-amd64
java-11-openjdk-amd64	   java-21-openjdk-amd64
apache-cassandra-4.1.5/bin/cassandra
Version: 3.5.3


In [3]:
import os, time
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
!apache-cassandra-4.1.5/bin/cassandra -R > cassandra.log 2>&1

from cassandra.cluster import Cluster
for i in range(24):
    try:
        session = Cluster(["127.0.0.1"]).connect()
        print("Cassandra ready")
        break
    except Exception:
        time.sleep(5)

from google.colab import drive
drive.mount("/content/drive")

Traceback (most recent call last):
  File "cassandra/cluster.py", line 3634, in cassandra.cluster.ControlConnection._reconnect_internal
  File "cassandra/cluster.py", line 3656, in cassandra.cluster.ControlConnection._try_connect
    connection = self._cluster.connection_factory(host.endpoint, is_control_connection=True)
  File "cassandra/cluster.py", line 1635, in cassandra.cluster.Cluster.connection_factory
    return self.connection_class.factory(endpoint, self.connect_timeout, *args, **kwargs)
  File "cassandra/connection.py", line 858, in cassandra.connection.Connection.factory
  File "/usr/local/lib/python3.13/dist-packages/cassandra/io/libevreactor.py", line 269, in __init__
    self._connect_socket()
    ~~~~~~~~~~~~~~~~~~~~^^
  File "cassandra/connection.py", line 963, in cassandra.connection.Connection._connect_socket
    raise socket.error(sockerr.errno, "Tried connecting to %s. Last error: %s" %
ConnectionRefusedError: [Errno 111] Tried connecting to [('127.0.0.1', 9042)]. 

Cassandra ready
Mounted at /content/drive


In [4]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.appName("YouTubeTrending")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())
spark

In [5]:
!ls /content/drive/MyDrive/BDA_Project/data

CA_category_id.json  GBvideos.csv	  US_category_id.json
CAvideos.csv	     IN_category_id.json  USvideos.csv
GB_category_id.json  INvideos.csv


In [6]:
import json
from functools import reduce

DATA = "/content/drive/MyDrive/BDA_Project/data"
countries = ["IN", "US", "GB", "CA"]

def load_country(c):
    df = (spark.read.option("header", True).option("multiLine", True)
          .option("escape", '"').csv(f"{DATA}/{c}videos.csv"))
    items = json.load(open(f"{DATA}/{c}_category_id.json", encoding="utf-8"))["items"]
    cat_df = spark.createDataFrame([(int(x["id"]), x["snippet"]["title"]) for x in items],
                                   ["category_id", "category"])
    return (df.withColumn("country", F.lit(c))
              .withColumn("category_id", F.col("category_id").cast("int"))
              .join(cat_df, "category_id", "left"))

raw = reduce(lambda a, b: a.unionByName(b), [load_country(c) for c in countries])

In [7]:
df = (raw
  .filter(F.col("video_id") != "#NAME?")
  .withColumn("trending_date", F.to_date("trending_date", "yy.dd.MM"))
  .withColumn("publish_time", F.to_timestamp("publish_time", "yyyy-MM-dd'T'HH:mm:ss.SSSX"))
  .withColumn("views", F.col("views").cast("long"))
  .withColumn("likes", F.col("likes").cast("long"))
  .withColumn("dislikes", F.col("dislikes").cast("long"))
  .withColumn("comment_count", F.col("comment_count").cast("long"))
  .withColumn("comments_disabled", F.col("comments_disabled").cast("boolean"))
  .withColumn("ratings_disabled", F.col("ratings_disabled").cast("boolean"))
  .fillna({"category": "Unknown"})
  .dropna(subset=["video_id", "trending_date", "views"])
  .dropDuplicates(["country", "video_id", "trending_date"])
  .withColumn("days_to_trend", F.datediff("trending_date", F.to_date("publish_time")))
  .withColumn("publish_hour", F.hour("publish_time"))
  .withColumn("publish_day", F.date_format("publish_time", "E")))

In [8]:
df.cache()
print("Total rows:", df.count())
df.select("country", "title", "category", "views", "days_to_trend").show(5, truncate=40)

Total rows: 152794
+-------+----------------------------------------+--------------+-------+-------------+
|country|                                   title|      category|  views|days_to_trend|
+-------+----------------------------------------+--------------+-------+-------------+
|     CA|              Cadet Kelly - Disneycember| Entertainment| 132359|            1|
|     CA|             TOP 10 DES JEUX 2017 (CLIP)|        Comedy|3289990|            4|
|     CA|[3rd Place] TwoFourSeven Co. | VIBE X...|People & Blogs|  15616|            2|
|     CA|      WE WANT TO TALK ABOUT OUR MARRIAGE|People & Blogs| 748374|            1|
|     CA|                    You're making WHAT?!|People & Blogs| 187743|            2|
+-------+----------------------------------------+--------------+-------+-------------+
only showing top 5 rows



In [9]:
df.groupBy("country").agg(
    F.count("*").alias("rows"),
    F.min("trending_date").alias("from"),
    F.max("trending_date").alias("to")
).show()

+-------+-----+----------+----------+
|country| rows|      from|        to|
+-------+-----+----------+----------+
|     CA|40881|2017-11-14|2018-06-14|
|     GB|38742|2017-11-14|2018-06-14|
|     US|40899|2017-11-14|2018-06-14|
|     IN|32272|2017-11-14|2018-06-14|
+-------+-----+----------+----------+



In [10]:
session.execute("""CREATE KEYSPACE IF NOT EXISTS youtube
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}""")
session.set_keyspace("youtube")

session.execute("""
CREATE TABLE IF NOT EXISTS trending_by_country (
  country text, trending_date date, video_id text,
  title text, channel_title text, category text, publish_time timestamp,
  views bigint, likes bigint, dislikes bigint, comment_count bigint,
  tags text, comments_disabled boolean, ratings_disabled boolean,
  days_to_trend int, publish_hour int, publish_day text,
  PRIMARY KEY ((country, trending_date), video_id)
)""")

cols = ["country", "trending_date", "video_id", "title", "channel_title", "category",
        "publish_time", "views", "likes", "dislikes", "comment_count", "tags",
        "comments_disabled", "ratings_disabled", "days_to_trend", "publish_hour", "publish_day"]

(df.select(cols).write.format("org.apache.spark.sql.cassandra")
   .options(keyspace="youtube", table="trending_by_country").mode("append").save())
print("Data Save in Cassandra")

Data Save in Cassandra


In [11]:
d = df.filter("country = 'IN'").select(F.max("trending_date")).first()[0]
print("Date:", d)

rows = session.execute(
    "SELECT title, channel_title, views FROM trending_by_country WHERE country = %s AND trending_date = %s LIMIT 5",
    ["IN", d])
for r in rows:
    print(r.views, "|", r.channel_title, "|", r.title)

Date: 2018-06-14
879924 | America's Got Talent | Front Pictures: You Have To See This Amazing Projection Act - America's Got Talent 2018
487236 | Raghav Digital | Akha Hind Ma Hetali | VIDEO | Geeta Rabari New Gujarati Song | Raghav Digital
100169 | NIROM मराठी | अज्या दाखवणार भय्याला हिसका|Lagira Zhala Jee|13 June 2018|Upcoming Twist|Nirom Marathi
1214342 | Angry Prash | What If PubG Was Made In India ?
1017216 | Technical Guruji | Nokia 8110 4G Unboxing & First Look - The Dumb 4G Phone 🔥🍌


In [12]:
tdf = (spark.read.format("org.apache.spark.sql.cassandra")
       .options(keyspace="youtube", table="trending_by_country").load())
tdf.createOrReplaceTempView("trending")
print("Cassandra rows:", tdf.count())

Cassandra rows: 152794


In [13]:
spark.catalog.cacheTable("trending")

def analyse(table, sql, *sort_cols):
    r = spark.sql(sql)
    r.orderBy(*sort_cols).show(10, truncate=40)
    (r.write.format("org.apache.spark.sql.cassandra")
       .options(keyspace="youtube", table=table).mode("append").save())
    print("Saved to Cassandra table:", table)

tables = [
 "CREATE TABLE IF NOT EXISTS category_summary (country text, category text, videos bigint, avg_views double, PRIMARY KEY ((country), category))",
 "CREATE TABLE IF NOT EXISTS channel_summary (country text, channel_title text, videos bigint, total_views bigint, PRIMARY KEY ((country), channel_title))",
 "CREATE TABLE IF NOT EXISTS time_to_trend (country text, days int, videos bigint, PRIMARY KEY ((country), days))",
 "CREATE TABLE IF NOT EXISTS trending_duration (country text, days_trending bigint, videos bigint, PRIMARY KEY ((country), days_trending))",
 "CREATE TABLE IF NOT EXISTS engagement_summary (country text, category text, like_pct double, dislike_pct double, comment_pct double, PRIMARY KEY ((country), category))",
 "CREATE TABLE IF NOT EXISTS upload_time_summary (country text, publish_day text, publish_hour int, videos bigint, PRIMARY KEY ((country), publish_day, publish_hour))",
 "CREATE TABLE IF NOT EXISTS tag_summary (country text, tag text, videos bigint, PRIMARY KEY ((country), tag))",
 "CREATE TABLE IF NOT EXISTS global_videos (video_id text PRIMARY KEY, title text, channel_title text, countries bigint, country_list text)",
]
for t in tables:
    session.execute(t)
print("8 summary tables ready")

8 summary tables ready


In [14]:
analyse("category_summary", """
SELECT country, category, COUNT(DISTINCT video_id) AS videos, ROUND(AVG(views), 0) AS avg_views
FROM trending GROUP BY country, category
""", F.desc("videos"))

+-------+---------------+------+---------+
|country|       category|videos|avg_views|
+-------+---------------+------+---------+
|     CA|  Entertainment|  8252|1016372.0|
|     IN|  Entertainment|  7563| 865021.0|
|     CA|News & Politics|  2943| 388221.0|
|     CA| People & Blogs|  2560| 786414.0|
|     IN|News & Politics|  2506| 350668.0|
|     CA|         Comedy|  1948| 982889.0|
|     CA|         Sports|  1932|1075584.0|
|     US|  Entertainment|  1622|2067649.0|
|     CA|          Music|  1568|3532525.0|
|     CA|  Howto & Style|  1272| 782684.0|
+-------+---------------+------+---------+
only showing top 10 rows

Saved to Cassandra table: category_summary


In [15]:
analyse("channel_summary", """
SELECT country, channel_title, COUNT(*) AS videos, SUM(max_views) AS total_views
FROM (SELECT country, channel_title, video_id, MAX(views) AS max_views
      FROM trending WHERE channel_title IS NOT NULL
      GROUP BY country, channel_title, video_id)
GROUP BY country, channel_title
""", F.desc("videos"))

+-------+------------------+------+-----------+
|country|     channel_title|videos|total_views|
+-------+------------------+------+-----------+
|     IN|Study IQ education|   169|    9637789|
|     CA|         VikatanTV|   159|  100703955|
|     IN|         Tarang TV|   158|   15823246|
|     CA|         SET India|   154|   24520467|
|     IN|    etvteluguindia|   153|   49921892|
|     IN|         VikatanTV|   153|  111174395|
|     IN|  Saregama TVShows|   152|   23068307|
|     CA|             MSNBC|   152|   37363303|
|     CA|       RadaanMedia|   149|   54525654|
|     CA|   The Young Turks|   147|   22276383|
+-------+------------------+------+-----------+
only showing top 10 rows

Saved to Cassandra table: channel_summary


In [16]:
analyse("time_to_trend", """
SELECT country, first_days AS days, COUNT(*) AS videos
FROM (SELECT country, video_id, MIN(days_to_trend) AS first_days
      FROM trending GROUP BY country, video_id)
WHERE first_days IS NOT NULL
GROUP BY country, first_days
""", F.desc("videos"))

+-------+----+------+
|country|days|videos|
+-------+----+------+
|     CA|   1| 19184|
|     IN|   1| 11636|
|     IN|   2|  3665|
|     US|   1|  2739|
|     CA|   2|  2439|
|     CA|   0|  2042|
|     US|   2|  1812|
|     GB|   1|  1232|
|     GB|   2|   720|
|     US|   3|   688|
+-------+----+------+
only showing top 10 rows

Saved to Cassandra table: time_to_trend


In [17]:
analyse("trending_duration", """
SELECT country, days_trending, COUNT(*) AS videos
FROM (SELECT country, video_id, COUNT(DISTINCT trending_date) AS days_trending
      FROM trending GROUP BY country, video_id)
GROUP BY country, days_trending
""", F.desc("videos"))

+-------+-------------+------+
|country|days_trending|videos|
+-------+-------------+------+
|     CA|            1| 14516|
|     IN|            1|  7824|
|     CA|            2|  5857|
|     IN|            2|  4168|
|     CA|            3|  2373|
|     IN|            3|  2285|
|     IN|            4|  1189|
|     CA|            4|  1053|
|     US|            6|   766|
|     US|            7|   710|
+-------+-------------+------+
only showing top 10 rows

Saved to Cassandra table: trending_duration


In [18]:
analyse("engagement_summary", """
SELECT country, category,
       ROUND(AVG(likes / views * 100), 3) AS like_pct,
       ROUND(AVG(dislikes / views * 100), 3) AS dislike_pct,
       ROUND(AVG(comment_count / views * 100), 3) AS comment_pct
FROM trending WHERE views > 0
GROUP BY country, category
""", F.desc("like_pct"))

+-------+--------------------+--------+-----------+-----------+
|country|            category|like_pct|dislike_pct|comment_pct|
+-------+--------------------+--------+-----------+-----------+
|     IN|Science & Technology|   7.784|      0.294|      2.614|
|     IN|      Pets & Animals|   7.636|      0.134|      0.887|
|     IN|              Comedy|    5.35|      0.244|      0.484|
|     IN|           Education|   5.315|      0.157|      0.667|
|     CA|              Comedy|   5.196|       0.13|      0.606|
|     CA|       Howto & Style|   5.033|      0.108|      0.729|
|     CA|              Gaming|    4.95|      0.128|      0.864|
|     CA|               Music|   4.856|      0.147|      0.446|
|     US|               Music|   4.757|      0.126|      0.386|
|     CA|      Pets & Animals|    4.63|      0.089|      0.906|
+-------+--------------------+--------+-----------+-----------+
only showing top 10 rows

Saved to Cassandra table: engagement_summary


In [19]:
analyse("upload_time_summary", """
SELECT country, publish_day, publish_hour, COUNT(DISTINCT video_id) AS videos
FROM trending WHERE publish_hour IS NOT NULL
GROUP BY country, publish_day, publish_hour
""", F.desc("videos"))

+-------+-----------+------------+------+
|country|publish_day|publish_hour|videos|
+-------+-----------+------------+------+
|     CA|        Tue|          16|   411|
|     CA|        Thu|          16|   399|
|     CA|        Fri|          16|   383|
|     CA|        Mon|          16|   349|
|     CA|        Sat|          16|   345|
|     CA|        Wed|          16|   330|
|     CA|        Thu|          17|   323|
|     CA|        Sun|          16|   296|
|     CA|        Fri|          17|   296|
|     CA|        Mon|          17|   291|
+-------+-----------+------------+------+
only showing top 10 rows

Saved to Cassandra table: upload_time_summary


In [20]:
analyse("tag_summary", """
SELECT country, tag, COUNT(DISTINCT video_id) AS videos
FROM (SELECT country, video_id, LOWER(TRIM(REGEXP_REPLACE(t, '"', ''))) AS tag
      FROM trending LATERAL VIEW EXPLODE(SPLIT(tags, '[|]')) x AS t)
WHERE tag != '' AND tag != '[none]'
GROUP BY country, tag
HAVING videos >= 3
""", F.desc("videos"))

+-------+------------+------+
|country|         tag|videos|
+-------+------------+------+
|     CA|       funny|  2142|
|     CA|      comedy|  1561|
|     CA|        news|  1403|
|     IN|      comedy|  1029|
|     CA|    politics|  1012|
|     IN|  television|   998|
|     IN|      serial|   993|
|     IN|full episode|   984|
|     IN|        show|   949|
|     IN|       funny|   891|
+-------+------------+------+
only showing top 10 rows

Saved to Cassandra table: tag_summary


In [21]:
analyse("global_videos", """
SELECT video_id, MAX(title) AS title, MAX(channel_title) AS channel_title,
       COUNT(DISTINCT country) AS countries,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(country))) AS country_list
FROM trending GROUP BY video_id
HAVING countries > 1
""", F.desc("countries"))

+-----------+----------------------------------------+--------------------------------------+---------+--------------+
|   video_id|                                   title|                         channel_title|countries|  country_list|
+-----------+----------------------------------------+--------------------------------------+---------+--------------+
|2Vv-BfVoq4g|Ed Sheeran - Perfect (Official Music ...|                            Ed Sheeran|        4|CA, GB, IN, US|
|425M4M3Opkk|Christopher Robin Official Teaser Tra...|                 Disney Movie Trailers|        4|CA, GB, IN, US|
|SbrOMrjhyvI|SERENITY Official Trailer (2018) Matt...|                             ONE Media|        4|CA, GB, IN, US|
|GBTV5ekQwWw|Stephanie McMahon sends Ronda Rousey ...|                                   WWE|        4|CA, GB, IN, US|
|5sEaYB4rLFQ|Fantastic Beasts: The Crimes of Grind...|                 Warner Bros. Pictures|        4|CA, GB, IN, US|
|6ZfuNTqbHE8|Marvel Studios' Avengers: Infinity 

In [22]:
%%writefile app.py
import datetime
import pandas as pd
import plotly.express as px
import streamlit as st
from cassandra.cluster import Cluster

st.set_page_config(page_title="YouTube Trending Dashboard", page_icon="🎬", layout="wide")
ROSE = ["#FDA4AF", "#FB7185", "#E11D48", "#9F1239"]
COUNTRIES = {"India": "IN", "USA": "US", "Great Britain": "GB", "Canada": "CA"}
LABELS = {"title": "Title", "channel_title": "Channel", "category": "Category", "views": "Views", "likes": "Likes",
          "comment_count": "Comments", "videos": "Trending videos", "total_views": "Total views",
          "countries": "Countries", "country_list": "Country list"}

@st.cache_resource
def connect():
    return Cluster(["127.0.0.1"]).connect("youtube")

@st.cache_data(ttl=600)
def query(cql, params=None):
    return pd.DataFrame(list(connect().execute(cql, params)))

def table(name, code):
    return query(f"SELECT * FROM {name} WHERE country = %s", (code,))

def style(fig, height=420):
    fig.update_layout(height=height, font_family="Poppins", title_font_size=17, coloraxis_showscale=False,
                      paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", margin=dict(l=10, r=20, t=55, b=10))
    return fig

def hbar(df, x, y, title, n=10, fmt=","):
    df = df.nlargest(n, x).sort_values(x)
    fig = px.bar(df, x=x, y=y, orientation="h", title=title, color=x, color_continuous_scale=ROSE)
    fig.update_traces(texttemplate="%{x:" + fmt + "}", textposition="outside", cliponaxis=False)
    fig.update_layout(xaxis_title=None, yaxis_title=None)
    fig.update_xaxes(range=[0, df[x].max() * 1.18], showgrid=False, showticklabels=False)
    return style(fig)

def vbar(df, x, y, title, xlabel):
    fig = px.bar(df, x=x, y=y, title=title, color=y, color_continuous_scale=ROSE)
    fig.update_layout(xaxis_title=xlabel, yaxis_title="Videos")
    return style(fig, 380)

def card(fig):
    with st.container(border=True):
        st.plotly_chart(fig, config={"displayModeBar": False})

def show_table(df):
    nums = {c: st.column_config.NumberColumn(format="localized") for c in ["Views", "Likes", "Comments", "Total views"]}
    st.dataframe(df.rename(columns=LABELS), hide_index=True, column_config=nums)

# Sidebar
st.sidebar.title("Trending Insights")
name = st.sidebar.selectbox("Choose a country", list(COUNTRIES))
code = COUNTRIES[name]
st.sidebar.divider()
st.sidebar.markdown("**How it works**  \n1. PySpark cleans the data  \n2. Cassandra stores data and results  \n3. This dashboard reads Cassandra")
st.sidebar.caption("Data: Kaggle Trending YouTube Video Statistics, Nov 2017 to Jun 2018")

# Data from Cassandra summary tables
cat, ch, eng = table("category_summary", code), table("channel_summary", code), table("engagement_summary", code)
ttt, dur = table("time_to_trend", code), table("trending_duration", code)
up, tags = table("upload_time_summary", code), table("tag_summary", code)

st.markdown(f"""
<div style="background: linear-gradient(120deg, #E11D48 0%, #FB7185 55%, #FDA4AF 100%);
            padding: 26px 32px; border-radius: 18px; color: white; margin-bottom: 20px;">
  <div style="font-size: 2.1rem; font-weight: 700; line-height: 1.2;">YouTube Trending Videos Analysis</div>
  <div style="font-size: 1.05rem; opacity: 0.95; margin-top: 6px;">{name} &nbsp;|&nbsp; Nov 2017 to Jun 2018 &nbsp;|&nbsp; Apache Spark + Cassandra</div>
</div>""", unsafe_allow_html=True)

k1, k2, k3, k4 = st.columns(4)
k1.metric("Trending videos", f"{dur.videos.sum():,}", border=True)
k2.metric("Channels", f"{len(ch):,}", border=True)
k3.metric("Trended within 1 day", f"{ttt[ttt.days <= 1].videos.sum() / ttt.videos.sum():.0%}", border=True)
k4.metric("Avg days on trending", f"{(dur.days_trending * dur.videos).sum() / dur.videos.sum():.1f}", border=True)

t1, t2, t3, t4, t5 = st.tabs(["Categories", "Channels & Tags", "Timing", "Search by date", "Global hits"])

with t1:
    a, b = st.columns(2)
    with a: card(hbar(cat, "videos", "category", "Trending videos by category"))
    with b: card(hbar(eng, "like_pct", "category", "Likes per 100 views", fmt=".2f"))

with t2:
    a, b = st.columns(2)
    with a: card(hbar(ch, "videos", "channel_title", "Top 10 channels"))
    with b: card(hbar(tags, "videos", "tag", "Top 10 tags"))
    search = st.text_input("Search a channel", placeholder="Type a channel name, e.g. zee")
    if search:
        found = ch[ch.channel_title.str.contains(search, case=False, na=False)]
        show_table(found.drop(columns="country").sort_values("videos", ascending=False))

with t3:
    a, b = st.columns(2)
    with a: card(vbar(ttt[ttt.days.between(0, 14)], "days", "videos", "Days from upload to trending", "Days after upload"))
    with b: card(vbar(dur, "days_trending", "videos", "Days a video stays on trending", "Days on trending list"))
    days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    heat = (up.pivot_table(index="publish_day", columns="publish_hour", values="videos", fill_value=0)
              .reindex(index=days, columns=range(24), fill_value=0))
    fig = px.imshow(heat, color_continuous_scale=["#FFF1F3", "#FECDD6"] + ROSE, aspect="auto",
                    labels=dict(x="Upload hour (UTC). IST = UTC + 5:30", y="", color="Videos"),
                    title="When were trending videos uploaded?")
    card(style(fig, 400))

with t4:
    day = st.date_input("Pick a date", datetime.date(2018, 1, 15),
                        min_value=datetime.date(2017, 11, 14), max_value=datetime.date(2018, 6, 14))
    cql = ("SELECT title, channel_title, category, views, likes, comment_count "
           "FROM trending_by_country WHERE country = %s AND trending_date = %s")
    vids = query(cql, (code, day))
    st.code(cql.replace("%s", "?").replace(" FROM", "\nFROM").replace(" WHERE", "\nWHERE"), language="sql")
    if vids.empty:
        st.info("No data for this date. Try another one.")
    else:
        st.badge(f"{len(vids)} videos were trending in {name} on {day:%d %b %Y}", icon="🎬", color="red")
        show_table(vids.sort_values("views", ascending=False))

with t5:
    glob = query("SELECT title, channel_title, countries, country_list FROM global_videos")
    n = st.slider("Trended in at least this many countries", 2, 4, 4)
    show = glob[glob.countries >= n].sort_values("countries", ascending=False)
    st.badge(f"{len(show)} videos trended in {n} or more countries", icon="🌍", color="red")
    show_table(show)

Writing app.py


In [23]:
import os, subprocess, time

theme = """[theme]
base = "light"
primaryColor = "#E11D48"
backgroundColor = "#FFF9FA"
secondaryBackgroundColor = "#FFEEF2"
textColor = "#3A1F2B"
font = "Poppins:https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&display=swap, sans-serif"
baseRadius = "12px"
borderColor = "#F6CFD8"
showWidgetBorder = true
redColor = "#E11D48"

[theme.sidebar]
backgroundColor = "#FFE4EA"
secondaryBackgroundColor = "#FFFFFF"
"""
for folder in ["/content/.streamlit", os.path.expanduser("~/.streamlit")]:
    os.makedirs(folder, exist_ok=True)
    open(f"{folder}/config.toml", "w").write(theme)

subprocess.run(["pkill", "-f", "streamlit run app.py"])
time.sleep(3)
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true --theme.base light > st.log 2>&1",
                 shell=True, cwd="/content", start_new_session=True)
time.sleep(8)
print(open("/content/st.log").read()[-500:])



2026-10-23 03:49:01.098 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.74.170.222:8501




In [24]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup streamlit run app.py --server.port 8501 --server.headless true > st.log 2>&1 &
!nohup ./cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1 &
!sleep 15 && grep -o "https://[a-z0-9-]*\.trycloudflare\.com" cf.log

https://scout-bradford-warnings-devon.trycloudflare.com
